# V135 - Cross-lineage consensus Top1 actions

Hipotesa tunggal: aksi yang dipilih DUA lineage independen (V132-analogue dan
mesin v127 seed baru) lebih presisi karena error independen saling cancel
(preseden V115 menang public via konsensus). Falsifier terdaftar: gain OOF
konsensus >= +0.00015 DAN benefit>harm DAN n>=3. Aksi paket = versi fresh run.
User protected V25 dikeluarkan dari paket bila masuk.


In [ ]:
import hashlib
import json
import os
import sys
import time
from hashlib import sha256
from pathlib import Path

import numpy as np
import pandas as pd

SMOKE = bool(int(os.getenv("V135_SMOKE", "0")))
SEED = 20260911
E5_ID = "intfloat/multilingual-e5-base"
NOMIC_ID = "nomic-ai/nomic-embed-text-v2-moe"
EXP_FILE = "d89714241c85dbaf93a5574550a5d9272c37b9b432c69fd5c5a2ff07fe690d84"

ROOT = Path("/home/athilla/Documents/IF_ITB/Lomba/IT-TODAY/intelligo-course-recommender")
for cand in (ROOT / "nb_code", Path("nb_code"), Path("../nb_code")):
    if (cand / "v127_pipeline.py").exists():
        sys.path.insert(0, str(cand))
        break
import v127_pipeline as vp

cfg = vp.Config(seed=SEED, smoke=SMOKE,
                outer_splits=2 if SMOKE else 5, inner_splits=2 if SMOKE else 3,
                calibration_trees=20 if SMOKE else 150, ranker_trees=20 if SMOKE else 150,
                word_features=5000, char_features=8000, threads=4,
                bootstraps=200 if SMOKE else 2000)

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data"), Path("../data"), ROOT / "data"]
DATA = next((c for c in CANDIDATES if (c / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
CH = next((c for c in [ROOT / "results/v132/v132/candidate_v132_DIAGNOSTIC.csv",
                       Path("results/v132/v132/candidate_v132_DIAGNOSTIC.csv"),
                       Path("../results/v132/v132/candidate_v132_DIAGNOSTIC.csv")] if c.exists()), None)
if CH is not None:
    assert sha256(CH.read_bytes()).hexdigest() == EXP_FILE, "champion file changed"
    print(f"protected base found: {CH}")
else:
    print("protected base not found: fail-close copy will be skipped")
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else ROOT / "results/v135")
OUT.mkdir(parents=True, exist_ok=True)
T = {}
print(f"SMOKE={SMOKE} DATA={DATA} OUT={OUT}")


## Load dan provenance cache

Sama seperti v134: bentuk dan pipeline dicek ulang.


In [ ]:
t0 = time.time()
data = vp.load_data(DATA)
n_train, n_test = data.n_train, data.n_test
docs = data.docs
print(f"groups={len(set(data.groups))} train={n_train} test={n_test}")

import torch
from sentence_transformers import SentenceTransformer
E5C = next(c for c in [ROOT / "results/v58/e5_users.npy", Path("results/v58/e5_users.npy"), Path("../results/v58/e5_users.npy")] if c.exists())
NOC = next(c for c in [ROOT / "results/v124/nomic_users.npy", Path("results/v124/nomic_users.npy"), Path("../results/v124/nomic_users.npy")] if c.exists())
E5U = np.asarray(np.load(E5C), dtype=np.float32)
NOU = np.asarray(np.load(NOC), dtype=np.float32)
assert E5U.shape == (n_train + n_test, 768) and NOU.shape == (n_train + n_test, 768)
probe_idx = [0, 7, n_train - 1, n_train, n_train + n_test - 1]
e5t = SentenceTransformer(E5_ID)
pe = np.asarray(e5t.encode(["query: " + docs[i] for i in probe_idx], normalize_embeddings=True))
ce = (pe * E5U[probe_idx]).sum(1)
print(f"e5 probe cosine: {np.round(ce, 6)}")
assert (ce > 0.999).all(), "e5 cache pipeline mismatch"
del e5t
torch.cuda.empty_cache()
from transformers import AutoModel, AutoTokenizer
nTok = AutoTokenizer.from_pretrained(NOMIC_ID, trust_remote_code=True)
nNet = AutoModel.from_pretrained(NOMIC_ID, trust_remote_code=True, torch_dtype=torch.float16, device_map={"": 0} if torch.cuda.is_available() else None).eval()
pn = []
with torch.no_grad():
    for i in probe_idx:
        e = nTok("search_query: " + docs[i], padding=True, truncation=True, max_length=256, return_tensors="pt")
        e = {k: v.to(nNet.device) for k, v in e.items()}
        h = nNet(**e).last_hidden_state
        m = e["attention_mask"].unsqueeze(-1).float()
        v = ((h * m).sum(1) / m.sum(1).clamp_min(1)).float()
        pn.append((v / v.norm()).cpu().numpy().ravel())
pn = np.array(pn)
cn = (pn * (NOU[probe_idx] / (np.linalg.norm(NOU[probe_idx], axis=1, keepdims=True) + 1e-12))).sum(1)
print(f"nomic probe cosine: {np.round(cn, 6)}")
assert (cn > 0.999).all(), "nomic cache pipeline mismatch"
del nNet
torch.cuda.empty_cache()
T["probe"] = round(time.time() - t0, 1)
print(f"provenance OK in {T['probe']}s")


## Silabus dari cache v127

Tanpa re-encode.


In [ ]:
ESC = next(c for c in [ROOT / "results/v127/e5_syll.npy", Path("results/v127/e5_syll.npy"), Path("../results/v127/e5_syll.npy")] if c.exists())
NSC = next(c for c in [ROOT / "results/v127/nomic_syll.npy", Path("results/v127/nomic_syll.npy"), Path("../results/v127/nomic_syll.npy")] if c.exists())
ES = np.asarray(np.load(ESC), dtype=np.float32)
NS = np.asarray(np.load(NSC), dtype=np.float32)
assert ES.shape == (17, 768) and NS.shape == (17, 768)
assert np.isfinite(ES).all() and np.isfinite(NS).all()
print("syllabi loaded", ES.shape, sha256(ES.tobytes()).hexdigest()[:16], sha256(NS.tobytes()).hexdigest()[:16])


## Fresh nested BASE run dengan ledger aksi penuh

Agree-policy per outer fold di lineage independen. Semua baris aksi disimpan.


In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

t0 = time.time()
labels = data.y.copy()
NM = len(data.modules)
outer = StratifiedGroupKFold(n_splits=cfg.outer_splits, shuffle=True, random_state=SEED)
splits = [(np.arange(n_train)[tr], np.arange(n_train)[va]) for tr, va in outer.split(np.zeros(n_train), data.strata, data.groups)]
engine = vp.Engine(data, labels, cfg, OUT / "_engine_cache")
ANCH = np.zeros((n_train, NM))
BASE = np.zeros((n_train, NM))
led_rows, fold_rows, thr_pick = [], [], []

def head_utils(fit, rows_u, anch_mat):
    ctrl = vp.control_features(data, rows_u, anch_mat)
    fe = np.concatenate([ctrl, vp.semantic_features(np.asarray(E5U)[rows_u], ES, anch_mat[rows_u])], -1)
    fn = np.concatenate([ctrl, vp.semantic_features(np.asarray(NOU)[rows_u], NS, anch_mat[rows_u])], -1)
    FC = ctrl.shape[-1]
    Uc = fit["control"].predict(ctrl.reshape(-1, FC)).reshape(-1, 3)
    Ue = fit["e5"].predict(fe.reshape(-1, fe.shape[-1])).reshape(-1, 3)
    Un = fit["nomic"].predict(fn.reshape(-1, fn.shape[-1])).reshape(-1, 3)
    return Uc, Ue, Un

for fi, (tr, va) in enumerate(splits):
    fit = vp.fit_selector(engine, tr, E5U, ES, NOU, NS)
    anchor = fit["anchor_full"]
    thr = fit["threshold"]
    thr_pick.append(thr)
    Uc_va, Ue_va, Un_va = head_utils(fit, va, anchor)
    order_va = vp.stable_order(anchor[va])
    tgt_va = vp.action_targets(labels[va], anchor[va])
    Bv = anchor[va].copy()
    picks = {}
    for j in range(len(va)):
        a1, a2 = int(Ue_va[j].argmax()), int(Un_va[j].argmax())
        if a1 == a2 and Ue_va[j, a1] > thr and Un_va[j, a2] > thr:
            Bv[j] = vp.apply_actions(anchor[va][j:j + 1], [a1 + 1], [True])[0]
            picks[j] = a1 + 1
    mg_va = np.ptp(anchor[va][np.arange(len(va))[:, None], order_va[:, :2]], axis=1)
    for j, u in enumerate(va):
        for k, r in enumerate((1, 2, 3)):
            led_fold_rows.append({"user_id": data.user_ids[u], "fold": fi,
                             "incumbent": data.modules[int(order_va[j, 0])],
                             "challenger": data.modules[int(order_va[j, r])],
                             "candidate_rank": r,
                             "Uc": float(Uc_va[j, k]), "Ue": float(Ue_va[j, k]), "Un": float(Un_va[j, k]),
                             "anchor_margin": float(mg_va[j]),
                             "selected_base": bool(picks.get(j) == r),
                             "realized_delta": float(tgt_va[j, k])})
    ANCH[va], BASE[va] = anchor[va], Bv
    fold_rows.append({"fold": fi, "thr": thr, "nB": len(picks),
                 "oof_anchor": float(vp.per_user_ndcg(labels[va], anchor[va]).mean()),
                 "oof_base": float(vp.per_user_ndcg(labels[va], Bv).mean())})
    print(f"fold {fi}: thr={thr} nB={len(picks)} anchor={fold_rows[-1]['oof_anchor']:.5f} base={fold_rows[-1]['oof_base']:.5f}", flush=True)
LED = pd.DataFrame(led_rows)
LED.to_csv(OUT / "v135_oof_ledger.csv", index=False)
print(f"ledger saved: {len(LED)} rows")
T["folds"] = round(time.time() - t0, 1)



## Konsensus lintas-lineage dan gate

Pengguna dipilih BOTH lineage. Aksi = versi fresh run. Gate: gain>=+0.00015,
benefit>harm, n>=3.


In [ ]:
t0 = time.time()
v132p = [ROOT / "results/v132/v132/v132_action_opportunity_ledger.csv",
         Path("results/v132/v132/v132_action_opportunity_ledger.csv"),
         Path("../results/v132/v132/v132_action_opportunity_ledger.csv")]
v132p = next(c for c in v132p if c.exists())
old = pd.read_csv(v132p)
S_old = set(old.loc[old["selected_v132"].astype(bool), "user_id"])
S_new = set(LED.loc[LED["selected_base"], "user_id"])
C = sorted(S_new & S_old)
print(f"fresh-selected users: {len(S_new)}, V132-selected: {len(S_old)}, consensus: {len(C)}")
con_rows = []
for u in C:
    r = LED[(LED["user_id"] == u) & (LED["selected_base"])].iloc[0]
    con_rows.append(r)
CON = pd.DataFrame(con_rows)
gain = float(CON["realized_delta"].sum() / n_train) if len(CON) else 0.0
ben = int((CON["realized_delta"] > 0).sum()) if len(CON) else 0
print(f"consensus n={len(CON)} gain {gain:+.6f} benefit {ben}/{len(CON)}")
if len(CON):
    print(CON[["user_id", "fold", "incumbent", "challenger", "candidate_rank", "realized_delta"]].to_string(index=False))
gate = bool(len(CON) >= 3 and gain >= 0.00015 and ben > len(CON) - ben)
print("GATE:", gate)
T["consensus"] = round(time.time() - t0, 1)


## Refit test, paket konsensus, file

Test agree-policy dari full-train. Paket = irisan dengan V132-test-5.
Kandidat hanya bila gate lolos.


In [ ]:
t0 = time.time()
fit_all = vp.fit_selector(engine, np.arange(n_train), E5U, ES, NOU, NS)
anchor_te = fit_all["anchor_full"][n_train:]
te_idx = np.arange(n_train, n_train + n_test)
Uc_t, Ue_t, Un_t = head_utils(fit_all, te_idx, fit_all["anchor_full"])
thrT = fit_all["threshold"]
order_te = vp.stable_order(anchor_te)
FINB = anchor_te.copy()
Tnew, picks_t = {}, {}
for j in range(n_test):
    a1, a2 = int(Ue_t[j].argmax()), int(Un_t[j].argmax())
    if a1 == a2 and Ue_t[j, a1] > thrT and Un_t[j, a2] > thrT:
        FINB[j] = vp.apply_actions(anchor_te[j:j + 1], [a1 + 1], [True])[0]
        Tnew[data.user_ids[n_train + j]] = a1 + 1
        picks_t[j] = a1 + 1
tled = []
for j in range(n_test):
    for k, r in enumerate((1, 2, 3)):
        tled.append({"user_id": data.user_ids[n_train + j],
                     "incumbent": data.modules[int(order_te[j, 0])],
                     "challenger": data.modules[int(order_te[j, r])],
                     "candidate_rank": r, "selected_base": bool(picks_t.get(j) == r)})
pd.DataFrame(tled).to_csv(OUT / "v135_test_ledger.csv", index=False)
v132t = [ROOT / "results/v132/v132/v132_test_action_opportunity_ledger.csv",
         Path("results/v132/v132/v132_test_action_opportunity_ledger.csv"),
         Path("../results/v132/v132/v132_test_action_opportunity_ledger.csv")] 
v132t = next(c for c in v132t if c.exists())
oldt = pd.read_csv(v132t)
T_old = set(oldt.loc[oldt["selected_v132"].astype(bool), "user_id"])
print(f"fresh test-selected: {len(Tnew)}, V132-test-5: {len(T_old)}")
prot = set(vp.V25_PROTECTED_USERS)
test_ids = data.user_ids[n_train:]
allowed = np.array([u not in prot for u in test_ids])
CT = sorted(set(Tnew) & T_old - prot)
print(f"consensus test users (non-protected): {CT}")
FINC = anchor_te.copy()
for u in CT:
    j = test_ids.index(u)
    FINC[j] = FINB[j]
adopt = bool(gate and len(CT) >= 1 and len(CT) <= 8)
print(f"adopt={adopt} (gate={gate}, nCT={len(CT)})")
if adopt:
    vp.write_submission(CH, OUT / "candidate_v135_primary.csv", FINC, True, allowed)
    print("wrote candidate_v135_primary.csv")
else:
    if CH is not None:
        vp.write_submission(CH, OUT / "champion_unchanged_v135.csv", FINC, False, allowed)
        assert sha256((OUT / "champion_unchanged_v135.csv").read_bytes()).hexdigest() == EXP_FILE
        print("gate failed: champion byte-copy written, hash-verified identical")
    else:
        print("gate failed: no champion file for fail-close copy")
REPORT = {"seed": SEED, "splits": cfg.outer_splits, "adopt": adopt, "gate": gate,
          "thr": thr_pick, "thr_test": thrT,
          "fresh_users": len(S_new), "v132_users": len(S_old), "consensus_users": C,
          "consensus_gain": gain, "consensus_benefit": ben,
          "fresh_test": len(Tnew), "v132_test": len(T_old), "consensus_test": CT,
          "fold_table": rows, "smoke": SMOKE, "timings": T, "audit_records": len(engine.fit_audit)}
(OUT / "v135_gate_report.json").write_text(json.dumps(REPORT, indent=1))
(OUT / "NOTES.md").write_text(f"# V135 cross-lineage consensus (seed {SEED})\nadopt={adopt} gate={gate} consensus_n={len(CON)} gain={gain:+.6f} benefit={ben} test_consensus={CT} smoke={SMOKE} timings={T}\n")
T["files"] = round(time.time() - t0, 1)
print(json.dumps(REPORT, indent=1)[:1200])
